In [39]:
import pandas as pd
import datetime
import warnings
import time
import numpy as np
from gf_ck_gen import save_google_flights_landing
from gf_req import requests_get_with_playwright_cookies
from gf_parser import parse_google_flights_html_text

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 30)
warnings.filterwarnings("ignore")

# Cookies Generation

Esegui questa cella quando vuoi aggiornare/salvare nuovi cookies. Le celle successive useranno automaticamente gli ultimi cookies salvati in `gf_landing_artifacts`, senza passare `result["cookies_path"]`.

In [2]:

url = "https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20to%20PMO%20from%20FCO%20on%202026-08-09%20oneway%20economy%20nonstops"
result = await save_google_flights_landing(url,headless=True,out_dir="gf_landing_artifacts",)
result

goto: https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20to%20PMO%20from%20FCO%20on%202026-08-09%20oneway%20economy%20nonstops
cookie banner clicked: True
final url before save: https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights+to+PMO+from+FCO+on+2026-08-09+oneway+economy+nonstops
{
  "input_url": "https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20to%20PMO%20from%20FCO%20on%202026-08-09%20oneway%20economy%20nonstops",
  "final_url": "https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights+to+PMO+from+FCO+on+2026-08-09+oneway+economy+nonstops",
  "cookie_banner_clicked": true,
  "cookies_path": "gf_landing_artifacts/20260619_135101_cookies.json",
  "cookies_count": 2,
  "html_path": "gf_landing_artifacts/20260619_135101_landing.html",
  "html_bytes": 2619710
}


{'input_url': 'https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20to%20PMO%20from%20FCO%20on%202026-08-09%20oneway%20economy%20nonstops',
 'final_url': 'https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights+to+PMO+from+FCO+on+2026-08-09+oneway+economy+nonstops',
 'cookie_banner_clicked': True,
 'cookies_path': 'gf_landing_artifacts/20260619_135101_cookies.json',
 'cookies_count': 2,
 'html_path': 'gf_landing_artifacts/20260619_135101_landing.html',
 'html_bytes': 2619710}

# Request Generation

V2: `cookies_path` è opzionale. Se non viene passato, `gf_req` usa l’ultimo file `*_cookies.json` trovato in `gf_landing_artifacts`. L’output `out` contiene direttamente anche `out["html"]`, cioè `response.text`.

In [13]:

url = "https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20from%20ROM%20to%20PMO%20on%202026-06-21%20oneway%20economy%20nonstops"
out = requests_get_with_playwright_cookies(url,out_html_path="gf_landing_artifacts/requests_get_response.html")

info={k: v for k, v in out.items() if k not in {"html", "response_text"}}
info

{'status_code': 200,
 'ok': True,
 'final_url': 'https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20from%20ROM%20to%20PMO%20on%202026-06-21%20oneway%20economy%20nonstops',
 'cookies_path': 'gf_landing_artifacts/20260619_135101_cookies.json',
 'cookies_source': 'latest_saved',
 'saved_path': 'gf_landing_artifacts/requests_get_response.html',
 'response_bytes': 2388471,
 'html_bytes': 2388471,
 'content_type': 'text/html; charset=utf-8',
 'encoding': 'utf-8'}

# Data Parser

V2: il parser riceve direttamente l’HTML in memoria da `out["html"]`, senza leggere il file locale.

In [23]:

df, info = parse_google_flights_html_text(
    out["html"],
    include_raw_label=False,)

df.head()


,result_index,flight_id,flight_id_source,section,price_eur,currency,airline,operated_by,carrier_code,flight_number,flight_departure_date,origin,destination,depart_time,arrive_time,...,stops_count,origin_airport_name,destination_airport_name,depart_day_text,arrive_day_text,emissions_kg_co2e,baggage_cabin_not_included,flight_segments_count,flight_segment_ids,tim_itinerary,travelimpactmodel_url,card_id,airport_codes_found,query_text,flight_segments_json
0,0,XZ2717,travelimpactmodel_itinerary,Voli più pertinenti,75,EUR,Aeroitalia,None,XZ,2717,2026-06-21,FCO,PMO,22:00,23:10,...,0,Aeroporto di Roma - Fiumicino Leonardo da Vinci,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",54.0,False,1,XZ2717,FCO-PMO-XZ-2717-20260621,https://www.travelimpactmodel.org/lookup/fligh...,QRCuqf,"FCO,PMO",Flights from ROM to PMO on 2026-06-21 oneway e...,"[{""origin"": ""FCO"", ""destination"": ""PMO"", ""carr..."
1,3,XZ2711,travelimpactmodel_itinerary,Altri voli,90,EUR,Aeroitalia,None,XZ,2711,2026-06-21,FCO,PMO,14:00,15:10,...,0,Aeroporto di Roma - Fiumicino Leonardo da Vinci,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",54.0,False,1,XZ2711,FCO-PMO-XZ-2711-20260621,https://www.travelimpactmodel.org/lookup/fligh...,e7Puqc,"FCO,PMO",Flights from ROM to PMO on 2026-06-21 oneway e...,"[{""origin"": ""FCO"", ""destination"": ""PMO"", ""carr..."
2,1,AZ1783,travelimpactmodel_itinerary,Voli più pertinenti,93,EUR,ITA,None,AZ,1783,2026-06-21,FCO,PMO,22:00,23:05,...,0,Aeroporto di Roma - Fiumicino Leonardo da Vinci,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",57.0,False,1,AZ1783,FCO-PMO-AZ-1783-20260621,https://www.travelimpactmodel.org/lookup/fligh...,cfRem,"FCO,PMO",Flights from ROM to PMO on 2026-06-21 oneway e...,"[{""origin"": ""FCO"", ""destination"": ""PMO"", ""carr..."
3,4,AZ1789,travelimpactmodel_itinerary,Altri voli,93,EUR,ITA,None,AZ,1789,2026-06-21,FCO,PMO,21:20,22:30,...,0,Aeroporto di Roma - Fiumicino Leonardo da Vinci,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",56.0,False,1,AZ1789,FCO-PMO-AZ-1789-20260621,https://www.travelimpactmodel.org/lookup/fligh...,qdCkqd,"FCO,PMO",Flights from ROM to PMO on 2026-06-21 oneway e...,"[{""origin"": ""FCO"", ""destination"": ""PMO"", ""carr..."
4,5,XZ2715,travelimpactmodel_itinerary,Altri voli,100,EUR,Aeroitalia,None,XZ,2715,2026-06-21,FCO,PMO,18:00,19:10,...,0,Aeroporto di Roma - Fiumicino Leonardo da Vinci,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",54.0,False,1,XZ2715,FCO-PMO-XZ-2715-20260621,https://www.travelimpactmodel.org/lookup/fligh...,XYiqHf,"FCO,PMO",Flights from ROM to PMO on 2026-06-21 oneway e...,"[{""origin"": ""FCO"", ""destination"": ""PMO"", ""carr..."


# Plan Gen

In [21]:
def url_compose(city_dep,city_arr,target_date):
    return f"https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20from%20{city_dep}%20to%20{city_arr}%20on%20{target_date}%20oneway%20economy%20nonstops"

def plan_gen(directory):
    plan_df=pd.read_excel(directory+"plan_sum26.xlsx")
    sigle_df=pd.read_excel(directory+"plan_sum26.xlsx",sheet_name='sigle')
    plan_df=pd.merge(plan_df,sigle_df.rename(columns={"city":"origin","sigla":"city_dep"}),on='origin')
    plan_df=pd.merge(plan_df,sigle_df.rename(columns={"city":"destination","sigla":"city_arr"}),on='destination')
    plan_df["url"]=plan_df.apply(lambda x:url_compose(x["city_dep"],x["city_arr"],x["target_date"]),axis=1)

    return plan_df

directory="plans/"
plan_df=plan_gen(directory)
plan_df.head()

,origin,destination,target_date,cluster,flag,city_dep,city_arr,url
0,Alghero,Milano,2026-08-09,from,sum26_ext,AHO,MIL,https://www.google.com/travel/flights?gl=IT&hl...
1,Alghero,Milano,2026-08-16,from,sum26_ext,AHO,MIL,https://www.google.com/travel/flights?gl=IT&hl...
2,Alghero,Milano,2026-08-23,from,sum26_ext,AHO,MIL,https://www.google.com/travel/flights?gl=IT&hl...
3,Alghero,Milano,2026-08-30,from,sum26_ext,AHO,MIL,https://www.google.com/travel/flights?gl=IT&hl...
4,Alghero,Roma,2026-08-09,from,sum26_ext,AHO,ROM,https://www.google.com/travel/flights?gl=IT&hl...


# Monitor Gen

In [40]:
def monitor_gen(plan_df):
    update_date=datetime.datetime.now().date()
    start_time=time.time()
    i=0
    for row in plan_df.to_dict(orient='records'):
        print("-------------------------","N.:",i,"origin:",row["origin"],"destination:",row["destination"],"target_date:",row["target_date"])
        url=row["url"]
        out = requests_get_with_playwright_cookies(url,out_html_path="gf_landing_artifacts/requests_get_response.html")
        loc_df,info=parse_google_flights_html_text(out["html"],include_raw_label=False,)
        print("N. results:",len(loc_df),"time:",np.round(time.time()-start_time,2))
        if len(loc_df)==0:
            "$$$$$$$$$$$$ Attention, no Data $$$$$$$$$$$$"
        loc_df.to_excel(f"output/{str(update_date)}_{row['city_dep']}_{row['city_arr']}_{row['target_date']}.xlsx")
        i+=1

        
monitor_gen(plan_df)  

------------------------- N.: 0 origin: Alghero destination: Milano target_date: 2026-08-09
N. results: 5 time: 2.67
------------------------- N.: 1 origin: Alghero destination: Milano target_date: 2026-08-16
N. results: 5 time: 3.98
------------------------- N.: 2 origin: Alghero destination: Milano target_date: 2026-08-23
N. results: 5 time: 5.48
------------------------- N.: 3 origin: Alghero destination: Milano target_date: 2026-08-30
N. results: 5 time: 7.06
------------------------- N.: 4 origin: Alghero destination: Roma target_date: 2026-08-09
N. results: 5 time: 8.67
------------------------- N.: 5 origin: Alghero destination: Roma target_date: 2026-08-16
N. results: 5 time: 10.14
------------------------- N.: 6 origin: Alghero destination: Roma target_date: 2026-08-23
N. results: 5 time: 11.68
------------------------- N.: 7 origin: Alghero destination: Roma target_date: 2026-08-30
N. results: 5 time: 13.42
------------------------- N.: 8 origin: Bari destination: Milano targ